# 04 – Model Evaluation

This notebook loads the held-out **test set** and generates predictions for every trained model so we can compare them side-by-side.

| # | Model | Type | Artifact |
|---|-------|------|----------|
| 1 | TF-IDF + Logistic Regression | Baseline (sklearn) | `models/tfidf_lr.pkl` |
| 2 | DistilBERT | Transformer | `models/distilbert-fakenews/` |
| 3 | RoBERTa | Transformer | `models/roberta-fakenews/` |
| 4 | DeBERTa | Transformer | `models/deberta-fakenews/` |

For each model the notebook stores `y_true`, `y_pred`, and `y_prob` in a single `results` dictionary for downstream metric calculation.

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import joblib
import torch
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# Add src/ to path so we can import the Phase 1 data pipeline
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from data import load_dataset, get_splits

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root : {PROJECT_ROOT}')
print(f'Device       : {DEVICE}')

## 2. Load Test Data

We call `get_splits()` and keep **only** `test_df` – the 10 % held-out partition that none of the models have seen during training or validation.

In [ ]:
df = load_dataset()
_, _, test_df = get_splits(df)

print(f'Test samples : {len(test_df):,}')
print(f'Label distribution:\n{test_df["label"].value_counts()}')

X_test = test_df['input_text']
y_true = test_df['label'].values

## 3. Define Model Registry

A list of dictionaries describing every model we want to evaluate.  
Each entry carries the info needed to load the model and run inference.

In [ ]:
MODEL_REGISTRY = [
    {
        'name': 'TF-IDF + LR (Baseline)',
        'key': 'baseline',
        'type': 'sklearn',
        'path': os.path.join(PROJECT_ROOT, 'models', 'tfidf_lr.pkl'),
    },
    {
        'name': 'DistilBERT',
        'key': 'distilbert',
        'type': 'transformer',
        'path': os.path.join(PROJECT_ROOT, 'models', 'distilbert-fakenews'),
    },
    {
        'name': 'RoBERTa',
        'key': 'roberta',
        'type': 'transformer',
        'path': os.path.join(PROJECT_ROOT, 'models', 'roberta-fakenews'),
    },
    {
        'name': 'DeBERTa',
        'key': 'deberta',
        'type': 'transformer',
        'path': os.path.join(PROJECT_ROOT, 'models', 'deberta-fakenews'),
    },
]

print(f'{len(MODEL_REGISTRY)} models registered for evaluation.')

## 4. Inference Helpers

Two helper functions – one for the sklearn baseline and one for HuggingFace transformers – that return `(y_pred, y_prob)` given the test texts.

In [ ]:
# ── Sklearn baseline inference ──────────────────────────────────────

def predict_sklearn(model_path, texts):
    """Load a joblib-serialised sklearn pipeline and return predictions."""
    pipeline = joblib.load(model_path)
    y_pred = pipeline.predict(texts)
    y_prob = pipeline.predict_proba(texts)[:, 1]   # P(label=1)
    return y_pred, y_prob


# ── Transformer inference ───────────────────────────────────────────

def predict_transformer(model_path, texts, batch_size=32, max_length=512):
    """
    Load a saved HuggingFace model + tokenizer and run batched inference.

    Returns
    -------
    y_pred : np.ndarray   – predicted class labels (0 or 1)
    y_prob : np.ndarray   – probability for the positive class (label=1)
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    all_logits = []
    text_list = list(texts)

    for i in tqdm(range(0, len(text_list), batch_size), desc='Inference'):
        batch_texts = text_list[i : i + batch_size]
        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**encodings)
        all_logits.append(outputs.logits.cpu().numpy())

    logits = np.concatenate(all_logits, axis=0)
    probs = softmax(logits, axis=1)
    y_pred = np.argmax(logits, axis=1)
    y_prob = probs[:, 1]  # P(label=1)
    return y_pred, y_prob

## 5. Run Inference for All Models

Loop over the registry, run the appropriate inference function, and store everything in a unified `results` dictionary.

```python
results[model_key] = {
    'name':   str,        # human-readable model name
    'y_true': np.ndarray, # ground-truth labels
    'y_pred': np.ndarray, # predicted class labels
    'y_prob': np.ndarray, # P(label=1) probabilities
}
```

In [ ]:
results = {}

for entry in MODEL_REGISTRY:
    name = entry['name']
    key  = entry['key']
    path = entry['path']

    # ── Guard: skip models that haven't been trained yet ────────────
    if not os.path.exists(path):
        print(f'⚠  {name:30s} → artifact not found, skipping.')
        continue

    print(f'▶  {name:30s} → loading from {path}')

    if entry['type'] == 'sklearn':
        y_pred, y_prob = predict_sklearn(path, X_test)
    else:
        y_pred, y_prob = predict_transformer(path, X_test)

    results[key] = {
        'name':   name,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

    print(f'   ✓ stored {len(y_pred):,} predictions\n')

print(f'\n── Done: {len(results)}/{len(MODEL_REGISTRY)} models evaluated ──')

## 6. Quick Sanity Check

Print a short summary for each evaluated model to confirm the data shapes and a rough accuracy before we move to the full metric suite.

In [ ]:
from sklearn.metrics import accuracy_score

for key, res in results.items():
    acc = accuracy_score(res['y_true'], res['y_pred'])
    print(
        f"{res['name']:30s}  |  "
        f"samples={len(res['y_true']):,}  |  "
        f"accuracy={acc:.4f}  |  "
        f"prob range=[{res['y_prob'].min():.4f}, {res['y_prob'].max():.4f}]"
    )

## 7. Disk Size

Calculate the on-disk footprint of every model artifact (single `.pkl` file **or** HuggingFace directory) and attach it to the `results` dictionary.

In [ ]:
def get_model_size_mb(path: str) -> float:
    """
    Return the total size of a model artifact in megabytes.

    - If `path` is a file (e.g. .pkl), return its size directly.
    - If `path` is a directory (HuggingFace checkpoint), sum all files
      recursively.
    """
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 ** 2)

    total_bytes = 0
    for dirpath, _, filenames in os.walk(path):
        for fname in filenames:
            fp = os.path.join(dirpath, fname)
            if os.path.isfile(fp):
                total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 ** 2)


# Attach disk size to each evaluated model
for entry in MODEL_REGISTRY:
    key  = entry['key']
    path = entry['path']
    if key not in results:
        continue
    size_mb = get_model_size_mb(path)
    results[key]['disk_size_mb'] = round(size_mb, 2)
    print(f"{results[key]['name']:30s}  |  {size_mb:>8.2f} MB")

## 8. Latency Benchmark

Measure single-sample inference latency for each model using `time.perf_counter()`.  
Protocol: **5 warm-up** iterations (discarded) followed by **100 timed** iterations.  
Latency is recorded separately for **CPU** and, when available, **GPU**.

In [ ]:
import time

WARMUP_ITERS = 5
BENCH_ITERS  = 100


# ── Sklearn single-sample callable ──────────────────────────────────

def _sklearn_single(pipeline, text):
    """Run a single sklearn prediction (no loading overhead)."""
    pipeline.predict([text])


# ── Transformer single-sample callable ──────────────────────────────

def _transformer_single(model, tokenizer, text, device, max_length=512):
    """Run a single transformer forward pass (no loading overhead)."""
    encodings = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        model(**encodings)
    # Ensure GPU work is finished before reading the clock
    if device.type == 'cuda':
        torch.cuda.synchronize()


# ── Generic benchmark harness ───────────────────────────────────────

def benchmark_latency(run_fn, warmup=WARMUP_ITERS, iters=BENCH_ITERS):
    """
    Benchmark a zero-argument callable.

    Parameters
    ----------
    run_fn  : callable  – the inference function to time (no args)
    warmup  : int       – warm-up iterations (discarded)
    iters   : int       – timed iterations to average over

    Returns
    -------
    float – mean latency in milliseconds
    """
    # Warm-up (caches, JIT compilation, etc.)
    for _ in range(warmup):
        run_fn()

    # Timed runs
    elapsed = []
    for _ in range(iters):
        t0 = time.perf_counter()
        run_fn()
        t1 = time.perf_counter()
        elapsed.append(t1 - t0)

    mean_ms = (sum(elapsed) / len(elapsed)) * 1000
    return round(mean_ms, 3)


print(f'Benchmark config  →  {WARMUP_ITERS} warm-up, {BENCH_ITERS} timed iterations')

## 9. Run Latency Benchmark

Pick one **representative article** from `test_df` and run the benchmark across all four models.  
For transformer models the benchmark is executed on **CPU first**, then on **GPU** (if available).

In [ ]:
# Pick a single representative test article (closest to median length)
lengths = test_df['input_text'].str.len()
median_idx = (lengths - lengths.median()).abs().idxmin()
sample_text = test_df.loc[median_idx, 'input_text']
print(f'Representative article index : {median_idx}')
print(f'Character length             : {len(sample_text):,}\n')

HAS_GPU = torch.cuda.is_available()

for entry in MODEL_REGISTRY:
    key  = entry['key']
    path = entry['path']

    if key not in results:
        continue

    name = results[key]['name']
    print(f'▶  Benchmarking {name} …')

    # ── Sklearn baseline ─────────────────────────────────────────────
    if entry['type'] == 'sklearn':
        pipeline = joblib.load(path)
        cpu_ms = benchmark_latency(
            lambda p=pipeline: _sklearn_single(p, sample_text)
        )
        results[key]['latency_cpu_ms'] = cpu_ms
        results[key]['latency_gpu_ms'] = None   # not applicable
        print(f'   CPU  : {cpu_ms:>8.3f} ms')
        print(f'   GPU  :      N/A (sklearn)\n')

    # ── Transformer models ───────────────────────────────────────────
    else:
        tokenizer = AutoTokenizer.from_pretrained(path)

        # ── CPU benchmark ────────────────────────────────────────────
        model_cpu = AutoModelForSequenceClassification.from_pretrained(path)
        model_cpu.to(torch.device('cpu'))
        model_cpu.eval()
        cpu_ms = benchmark_latency(
            lambda m=model_cpu, t=tokenizer: _transformer_single(
                m, t, sample_text, torch.device('cpu')
            )
        )
        results[key]['latency_cpu_ms'] = cpu_ms
        print(f'   CPU  : {cpu_ms:>8.3f} ms')
        del model_cpu   # free memory before GPU allocation

        # ── GPU benchmark (if available) ─────────────────────────────
        if HAS_GPU:
            model_gpu = AutoModelForSequenceClassification.from_pretrained(path)
            model_gpu.to(torch.device('cuda'))
            model_gpu.eval()
            gpu_ms = benchmark_latency(
                lambda m=model_gpu, t=tokenizer: _transformer_single(
                    m, t, sample_text, torch.device('cuda')
                )
            )
            results[key]['latency_gpu_ms'] = gpu_ms
            print(f'   GPU  : {gpu_ms:>8.3f} ms\n')
            del model_gpu
            torch.cuda.empty_cache()
        else:
            results[key]['latency_gpu_ms'] = None
            print(f'   GPU  :      N/A (no CUDA)\n')

print('── Latency benchmarking complete ──')

## 10. Operational Metrics Summary

Print a consolidated table of disk size and latency for every evaluated model.

In [ ]:
import pandas as pd

rows = []
for key, res in results.items():
    rows.append({
        'Model':            res['name'],
        'Disk Size (MB)':   res.get('disk_size_mb'),
        'CPU Latency (ms)': res.get('latency_cpu_ms'),
        'GPU Latency (ms)': res.get('latency_gpu_ms'),
    })

ops_df = pd.DataFrame(rows)
display(ops_df.style.format({
    'Disk Size (MB)':   '{:.2f}',
    'CPU Latency (ms)': '{:.3f}',
    'GPU Latency (ms)': lambda v: f'{v:.3f}' if v is not None else 'N/A',
}).set_caption('Operational Metrics – Disk Size & Latency'))

## 11. Classification Metrics

Calculate **Accuracy**, **Macro F1**, **ROC-AUC**, and **Log Loss** for every evaluated model and merge them with the operational metrics already in `results`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

for key, res in results.items():
    yt, yp, yprob = res['y_true'], res['y_pred'], res['y_prob']

    res['accuracy']  = round(accuracy_score(yt, yp), 5)
    res['macro_f1']  = round(f1_score(yt, yp, average='macro'), 5)
    res['roc_auc']   = round(roc_auc_score(yt, yprob), 5)
    res['log_loss']  = round(log_loss(yt, yprob), 5)

    print(
        f"{res['name']:30s}  |  "
        f"Acc={res['accuracy']:.4f}  "
        f"F1={res['macro_f1']:.4f}  "
        f"AUC={res['roc_auc']:.4f}  "
        f"LogLoss={res['log_loss']:.4f}"
    )

## 12. Export JSON Report

Combine all statistical and operational metrics into a single JSON object and write it to `results/final_comparison.json`.

In [ ]:
import json

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Build a JSON-safe version (drop numpy arrays)
export = {}
for key, res in results.items():
    export[key] = {
        'model_name':      res['name'],
        'accuracy':        res['accuracy'],
        'macro_f1':        res['macro_f1'],
        'roc_auc':         res['roc_auc'],
        'log_loss':        res['log_loss'],
        'disk_size_mb':    res.get('disk_size_mb'),
        'latency_cpu_ms':  res.get('latency_cpu_ms'),
        'latency_gpu_ms':  res.get('latency_gpu_ms'),
    }

json_path = os.path.join(RESULTS_DIR, 'final_comparison.json')
with open(json_path, 'w') as f:
    json.dump(export, f, indent=4)

print(f'Saved → {json_path}')
print(json.dumps(export, indent=4))

## 13. Combined ROC Curve

Plot all models on a single ROC axis with their AUC scores visible in the legend.  
Saved to `figures/roc_all_models.png`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

FIGURES_DIR = os.path.join(PROJECT_ROOT, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Color palette
COLORS = {
    'baseline':   '#636EFA',
    'distilbert': '#EF553B',
    'roberta':    '#00CC96',
    'deberta':    '#AB63FA',
}

fig, ax = plt.subplots(figsize=(9, 7))

for key, res in results.items():
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_prob'])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(
        fpr, tpr,
        color=COLORS.get(key, 'grey'),
        lw=2,
        label=f"{res['name']}  (AUC = {roc_auc_val:.4f})",
    )

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random Baseline')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves – All Models', fontsize=15, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()

roc_path = os.path.join(FIGURES_DIR, 'roc_all_models.png')
fig.savefig(roc_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {roc_path}')

## 14. Confusion Matrices

Generate and save an individual confusion-matrix heatmap for each model to `figures/confusion_<model_key>.png`.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

for key, res in results.items():
    cm = confusion_matrix(res['y_true'], res['y_pred'])

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Fake (0)', 'Real (1)'],
        yticklabels=['Fake (0)', 'Real (1)'],
        ax=ax,
    )
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(f"Confusion Matrix – {res['name']}", fontsize=14, fontweight='bold')
    fig.tight_layout()

    cm_path = os.path.join(FIGURES_DIR, f'confusion_{key}.png')
    fig.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved → {cm_path}\n')

## 15. Latency Benchmark Chart

Side-by-side bar chart comparing CPU and GPU inference latency for all models.  
Saved to `figures/latency_benchmark.png`.

In [ ]:
model_names = [res['name'] for res in results.values()]
cpu_times   = [res.get('latency_cpu_ms', 0) or 0 for res in results.values()]
gpu_times   = [res.get('latency_gpu_ms', 0) or 0 for res in results.values()]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars_cpu = ax.bar(x - width/2, cpu_times, width, label='CPU', color='#636EFA')
bars_gpu = ax.bar(x + width/2, gpu_times, width, label='GPU', color='#EF553B')

# Value labels
for bar in bars_cpu:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}',
                ha='center', va='bottom', fontsize=9)
for bar in bars_gpu:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}',
                ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Latency (ms / item)', fontsize=13)
ax.set_title('Single-Sample Inference Latency', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

lat_path = os.path.join(FIGURES_DIR, 'latency_benchmark.png')
fig.savefig(lat_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {lat_path}')

## 16. Final Leaderboard

Display the complete `final_comparison.json` data as a formatted DataFrame so the best production model can be easily identified.

In [ ]:
# Reload the exported JSON to prove round-trip integrity
with open(os.path.join(RESULTS_DIR, 'final_comparison.json')) as f:
    final_data = json.load(f)

leader_df = pd.DataFrame(final_data).T

# Highlight best values
display(
    leader_df.style
    .format({
        'accuracy':       '{:.4f}',
        'macro_f1':       '{:.4f}',
        'roc_auc':        '{:.4f}',
        'log_loss':       '{:.4f}',
        'disk_size_mb':   '{:.2f}',
        'latency_cpu_ms': '{:.2f}',
        'latency_gpu_ms': lambda v: f'{v:.2f}' if v is not None else 'N/A',
    })
    .highlight_max(subset=['accuracy', 'macro_f1', 'roc_auc'], color='#d4edda')
    .highlight_min(subset=['log_loss', 'disk_size_mb', 'latency_cpu_ms'], color='#d4edda')
    .set_caption('Final Model Comparison – Leaderboard')
)

# Identify best model by Macro F1
best_key = leader_df['macro_f1'].astype(float).idxmax()
print(f'\n🏆  Best model (by Macro F1): {final_data[best_key]["model_name"]}')